# 06 · Calibration, conformal guarantees & robustness
1. Split held-out data into **calibration** and **test** (by patient/source where possible).
2. Temperature-scale, then fit class-conditional (Mondrian) conformal thresholds for the *suspicious* class at α = 0.05.
3. Verify empirical recall ≥ 1 − α on the **test** split, per subgroup.
4. MedMNIST-C corruption robustness and OOD detection (energy score).

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
SEED = 13

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image

from rarecare.vision.model import ORGAN_HEADS, load_checkpoint
from rarecare.vision.preprocess import to_model_input

model = load_checkpoint("checkpoints/vision/convnext_tiny.pt", device)
m = pd.read_csv("data/manifests/vision.csv")
held = m[(m.split == "val") & (m.label >= 0)].sample(frac=1, random_state=13)
cal, test = held.iloc[: len(held) // 2], held.iloc[len(held) // 2 :]

def predict(df):
    rows = []
    for r in df.itertuples():
        x = torch.from_numpy(to_model_input(Image.open(r.path).convert("RGB")))[None].to(device)
        with torch.no_grad():
            a = model(x)[f"evidence_{r.modality}"][0] + 1
        rows.append({"cancer": ORGAN_HEADS[r.modality], "p_suspicious": float(a[1] / a.sum()), "label": r.label, "group": r.source})
    return pd.DataFrame(rows)

cal_p, test_p = predict(cal), predict(test)
cal_p.to_csv("checkpoints/calib_preds.csv", index=False)

In [ ]:
!python -m training.calibrate --preds checkpoints/calib_preds.csv --alpha 0.05 --out checkpoints/conformal.json

In [ ]:
# Verify on the untouched test split
import json

from rarecare.uncertainty.conformal import PositiveClassConformal, empirical_recall

conf = json.load(open("checkpoints/conformal.json"))
for cancer, g in test_p.groupby("cancer"):
    cp = PositiveClassConformal(alpha=conf[cancer]["alpha"], thresholds=conf[cancer]["thresholds"], n_calibration=conf[cancer]["n"])
    flags = np.array([cp.flag(p, s) for p, s in zip(g.p_suspicious, g.group)])
    print(cancer, "test recall", round(empirical_recall(flags, g.label.values), 3), "flag rate", round(flags.mean(), 3))
    for src, gg in g.groupby("group"):
        f = np.array([cp.flag(p, s) for p, s in zip(gg.p_suspicious, gg.group)])
        print("   ", src, "recall", round(empirical_recall(f, gg.label.values), 3), "n_pos", int((gg.label == 1).sum()))

In [ ]:
# Calibration diagnostics
from rarecare.uncertainty.calibration import brier_score, expected_calibration_error

for cancer, g in test_p.groupby("cancer"):
    probs = np.stack([1 - g.p_suspicious, g.p_suspicious], 1)
    print(cancer, "ECE", round(expected_calibration_error(probs, g.label.values), 4), "Brier", round(brier_score(probs, g.label.values), 4))

## OOD detection: energy score on in-scope vs. unsupported / non-medical validation images

In [ ]:
from sklearn.metrics import roc_auc_score

from rarecare.vision.ood import energy_score

val = m[m.split == "val"].sample(n=min(2000, (m.split == "val").sum()), random_state=13)
e, ood = [], []
for r in val.itertuples():
    x = torch.from_numpy(to_model_input(Image.open(r.path).convert("RGB")))[None].to(device)
    with torch.no_grad():
        lg = model(x)["router_logits"][0].cpu().numpy().astype(np.float64)
    e.append(energy_score(lg)); ood.append(int(r.modality not in ORGAN_HEADS))
print("OOD AUROC (energy)", round(roc_auc_score(ood, e), 4))